In [3]:
import sys
from datetime import datetime
from html import escape
from pathlib import Path

import ipywidgets as widgets
from IPython.display import clear_output, display


# =========================================================
# 1. FIND PROJECT FOLDER AND IMPORT MODULES
# =========================================================

CURRENT_FOLDER = Path.cwd().resolve()

if (CURRENT_FOLDER / "src").is_dir():
    PROJECT_ROOT = CURRENT_FOLDER

elif (CURRENT_FOLDER.parent / "src").is_dir():
    PROJECT_ROOT = CURRENT_FOLDER.parent

else:
    raise FileNotFoundError(
        "Could not find the project folder. "
        "Open this notebook from the notebooks folder inside "
        "CUSTOM-AI-PDF-READER."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pdf_service import (
    get_page_count,
    open_pdf,
    render_page,
    search_document as search_pdf_document
)

from src.reader_state import reader_state

from src.storage_service import (
    load_document_data,
    save_document_data
)


# =========================================================
# 2. OPEN PDF AND RESTORE SAVED DATA
# =========================================================

PDF_PATH = PROJECT_ROOT / "test.pdf"

if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"PDF was not found: {PDF_PATH}\n\n"
        "Put the PDF in your CUSTOM-AI-PDF-READER folder "
        "and name it test.pdf."
    )

document = open_pdf(PDF_PATH)

DOCUMENT_ID = str(PDF_PATH.resolve())

saved_document_data = load_document_data(
    document_id=DOCUMENT_ID,
    pdf_path=PDF_PATH
)

saved_last_page = saved_document_data.get("last_page", 0)

if not 0 <= saved_last_page < get_page_count(document):
    saved_last_page = 0

reader_state["current_page"] = saved_last_page
reader_state["zoom_dpi"] = 120
reader_state["search_results"] = []
reader_state["search_index"] = 0
reader_state["bookmarks"] = saved_document_data.get("bookmarks", [])
reader_state["notes"] = saved_document_data.get("notes", [])

print("PDF:", PDF_PATH.name)
print("Total pages:", get_page_count(document))
print("Restored page:", reader_state["current_page"] + 1)
print("Bookmarks:", len(reader_state["bookmarks"]))
print("Notes:", len(reader_state["notes"]))


# =========================================================
# 3. SAVE HELPER
# =========================================================

def save_current_reader_data():
    save_document_data(
        document_id=DOCUMENT_ID,
        pdf_path=PDF_PATH,
        bookmarks=reader_state["bookmarks"],
        notes=reader_state["notes"],
        last_page=reader_state["current_page"]
    )


# =========================================================
# 4. CREATE READER INTERFACE
# =========================================================

previous_button = widgets.Button(description="◀ Previous")
next_button = widgets.Button(description="Next ▶")

zoom_out_button = widgets.Button(description="− Zoom")
zoom_in_button = widgets.Button(description="+ Zoom")

page_input = widgets.BoundedIntText(
    value=reader_state["current_page"] + 1,
    min=1,
    max=get_page_count(document),
    description="Page:"
)

go_button = widgets.Button(description="Go")

page_label = widgets.Label()
zoom_label = widgets.Label()

search_input = widgets.Text(
    placeholder="Type a word or phrase...",
    description="Search:"
)

search_button = widgets.Button(
    description="Search",
    button_style="info"
)

previous_result_button = widgets.Button(
    description="◀ Previous result"
)

next_result_button = widgets.Button(
    description="Next result ▶"
)

search_status = widgets.Label(value="Search is ready.")

bookmark_label_input = widgets.Text(
    placeholder="Example: Important result",
    description="Label:"
)

add_bookmark_button = widgets.Button(
    description="☆ Add bookmark",
    button_style="success"
)

bookmark_status = widgets.Label(value="")

note_input = widgets.Textarea(
    placeholder="Write a note for the current page...",
    description="Note:",
    layout=widgets.Layout(width="650px", height="90px")
)

add_note_button = widgets.Button(
    description="Save note",
    button_style="success"
)

note_status = widgets.Label(value="")

page_output = widgets.Output()
bookmarks_output = widgets.Output()
notes_output = widgets.Output()

navigation_bar = widgets.HBox([
    previous_button,
    next_button,
    zoom_out_button,
    zoom_in_button,
    page_input,
    go_button,
    page_label,
    zoom_label
])

search_bar = widgets.HBox([
    search_input,
    search_button,
    previous_result_button,
    next_result_button,
    search_status
])

bookmark_bar = widgets.HBox([
    bookmark_label_input,
    add_bookmark_button,
    bookmark_status
])

note_bar = widgets.HBox([
    note_input,
    add_note_button,
    note_status
])


# =========================================================
# 5. REFRESH FUNCTIONS
# =========================================================

def refresh_reader():
    current_page = reader_state["current_page"]
    dpi = reader_state["zoom_dpi"]

    page_input.value = current_page + 1

    page_label.value = (
        f"Page {current_page + 1} / {get_page_count(document)}"
    )

    zoom_label.value = f"Render: {dpi} DPI"

    previous_button.disabled = current_page == 0

    next_button.disabled = (
        current_page == get_page_count(document) - 1
    )

    highlights = None
    results = reader_state["search_results"]

    if results:
        current_result = results[reader_state["search_index"]]

        if current_result["page_number"] == current_page:
            highlights = current_result["rectangles"]

    with page_output:
        clear_output(wait=True)

        page_image = render_page(
            document=document,
            page_number=current_page,
            dpi=dpi,
            highlight_rectangles=highlights
        )

        display(page_image)


def refresh_bookmarks():
    with bookmarks_output:
        clear_output(wait=True)

        bookmarks = reader_state["bookmarks"]

        if not bookmarks:
            print("No bookmarks saved.")
            return

        print("Saved bookmarks:")

        for index, bookmark in enumerate(bookmarks):
            page_number = bookmark["page_number"]
            label = bookmark["label"]

            open_button = widgets.Button(
                description=f"Page {page_number + 1}: {label}",
                layout=widgets.Layout(width="400px")
            )

            delete_button = widgets.Button(
                description="Delete",
                button_style="danger",
                layout=widgets.Layout(width="90px")
            )

            def open_bookmark(button, target_page=page_number):
                change_page(target_page)

            def delete_bookmark(button, bookmark_index=index):
                reader_state["bookmarks"].pop(bookmark_index)

                save_current_reader_data()

                refresh_bookmarks()

            open_button.on_click(open_bookmark)
            delete_button.on_click(delete_bookmark)

            display(
                widgets.HBox([
                    open_button,
                    delete_button
                ])
            )


def refresh_notes():
    with notes_output:
        clear_output(wait=True)

        notes = reader_state["notes"]

        if not notes:
            print("No notes saved.")
            return

        print("Saved notes:")

        for index, note in enumerate(notes):
            page_number = note["page_number"]
            note_text = note["text"]
            created_at = note["created_at"]

            open_button = widgets.Button(
                description=f"Page {page_number + 1}",
                layout=widgets.Layout(width="90px")
            )

            delete_button = widgets.Button(
                description="Delete",
                button_style="danger",
                layout=widgets.Layout(width="90px")
            )

            safe_note_text = escape(note_text).replace("\n", "<br>")

            note_details = widgets.HTML(
                value=(
                    f"<b>Page {page_number + 1}</b> "
                    f"<span style='color: #666;'>"
                    f"({escape(created_at)})"
                    f"</span><br>"
                    f"{safe_note_text}"
                ),
                layout=widgets.Layout(width="550px")
            )

            def open_note(button, target_page=page_number):
                change_page(target_page)

            def delete_note(button, note_index=index):
                reader_state["notes"].pop(note_index)

                save_current_reader_data()

                refresh_notes()

            open_button.on_click(open_note)
            delete_button.on_click(delete_note)

            display(
                widgets.HBox([
                    open_button,
                    note_details,
                    delete_button
                ])
            )


# =========================================================
# 6. NAVIGATION AND ZOOM
# =========================================================

def change_page(new_page):
    if not 0 <= new_page < get_page_count(document):
        return

    reader_state["current_page"] = new_page

    save_current_reader_data()

    refresh_reader()


def show_previous_page(button):
    change_page(reader_state["current_page"] - 1)


def show_next_page(button):
    change_page(reader_state["current_page"] + 1)


def go_to_page(button):
    change_page(page_input.value - 1)


def zoom_out(button):
    if reader_state["zoom_dpi"] > 72:
        reader_state["zoom_dpi"] -= 24

        refresh_reader()


def zoom_in(button):
    if reader_state["zoom_dpi"] < 240:
        reader_state["zoom_dpi"] += 24

        refresh_reader()


# =========================================================
# 7. SEARCH
# =========================================================

def run_search(button=None):
    term = search_input.value.strip()

    if not term:
        reader_state["search_results"] = []
        reader_state["search_index"] = 0

        search_status.value = "Enter text to search."

        refresh_reader()

        return

    reader_state["search_results"] = search_pdf_document(
        document,
        term
    )

    reader_state["search_index"] = 0

    if not reader_state["search_results"]:
        search_status.value = f'No results for "{term}".'

        refresh_reader()

        return

    show_current_result()


def show_current_result():
    results = reader_state["search_results"]

    if not results:
        return

    result = results[reader_state["search_index"]]

    reader_state["current_page"] = result["page_number"]

    save_current_reader_data()

    search_status.value = (
        f'Result {reader_state["search_index"] + 1} '
        f'/ {len(results)} '
        f'on page {result["page_number"] + 1}'
    )

    refresh_reader()


def show_next_result(button):
    results = reader_state["search_results"]

    if not results:
        search_status.value = "Search for a word first."

        return

    reader_state["search_index"] = (
        reader_state["search_index"] + 1
    ) % len(results)

    show_current_result()


def show_previous_result(button):
    results = reader_state["search_results"]

    if not results:
        search_status.value = "Search for a word first."

        return

    reader_state["search_index"] = (
        reader_state["search_index"] - 1
    ) % len(results)

    show_current_result()


# =========================================================
# 8. BOOKMARKS
# =========================================================

def add_bookmark(button):
    current_page = reader_state["current_page"]

    label = bookmark_label_input.value.strip()

    if not label:
        label = f"Page {current_page + 1}"

    already_exists = any(
        bookmark["page_number"] == current_page
        and bookmark["label"] == label
        for bookmark in reader_state["bookmarks"]
    )

    if already_exists:
        bookmark_status.value = "That bookmark already exists."

        return

    reader_state["bookmarks"].append({
        "page_number": current_page,
        "label": label
    })

    save_current_reader_data()

    bookmark_label_input.value = ""

    bookmark_status.value = (
        f"Saved bookmark for page {current_page + 1}."
    )

    refresh_bookmarks()


# =========================================================
# 9. NOTES
# =========================================================

def add_note(button):
    note_text = note_input.value.strip()

    if not note_text:
        note_status.value = "Write a note before saving."

        return

    reader_state["notes"].append({
        "page_number": reader_state["current_page"],
        "text": note_text,
        "created_at": datetime.now().strftime(
            "%Y-%m-%d %H:%M"
        )
    })

    save_current_reader_data()

    note_input.value = ""

    note_status.value = (
        f"Saved note for page "
        f"{reader_state['current_page'] + 1}."
    )

    refresh_notes()


# =========================================================
# 10. CONNECT BUTTONS
# =========================================================

previous_button.on_click(show_previous_page)
next_button.on_click(show_next_page)

go_button.on_click(go_to_page)

zoom_out_button.on_click(zoom_out)
zoom_in_button.on_click(zoom_in)

search_button.on_click(run_search)
next_result_button.on_click(show_next_result)
previous_result_button.on_click(show_previous_result)

add_bookmark_button.on_click(add_bookmark)
add_note_button.on_click(add_note)


# =========================================================
# 11. DISPLAY APP
# =========================================================

display(navigation_bar)
display(search_bar)
display(page_output)

display(bookmark_bar)
display(bookmarks_output)

display(note_bar)
display(notes_output)

refresh_reader()
refresh_bookmarks()
refresh_notes()

PDF: test.pdf
Total pages: 23
Restored page: 1
Bookmarks: 0
Notes: 0


Output()

Output()

Output()